# E06 California Housing：空间分组 vs 随机划分

> Issue #9 / Epic #6 / 路线图 E06

## 你将学到
1. 地理邻近样本相关 → 随机划分会泄漏空间信息
2. 用经纬度分桶 / GroupKFold 对比随机划分的乐观偏差
3. 不同房价区间的误差结构不同

## 数据
`sklearn.datasets.fetch_california_housing`


In [ ]:
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE

JE.no_warning()
V.show_env_info()


## 1. 加载数据


In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print(df.shape)
df.head()


## 2. EDA


In [ ]:
from wanglaoshi import DataAnalyzer

eda_df = df.sample(5000, random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
display(analyzer.basic_statistics())
print(df[['Latitude', 'Longitude', 'MedHouseVal']].describe())


## 3. 对比：随机划分 vs 空间分桶划分


In [ ]:
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

feature_cols = [c for c in df.columns if c != 'MedHouseVal']
X = df[feature_cols]
y = df['MedHouseVal']

# A) 随机划分
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
model = RandomForestRegressor(
    n_estimators=50 if QUICK else 150, random_state=RANDOM_STATE, n_jobs=-1, max_depth=12
)
model.fit(Xtr_r, ytr_r)
pred_r = model.predict(Xte_r)
print('RANDOM MAE', mean_absolute_error(yte_r, pred_r),
      'RMSE', mean_squared_error(yte_r, pred_r, squared=False))

# B) 空间分桶：把 lat/lon 量化成网格 group
lat_bin = pd.cut(df['Latitude'], bins=10, labels=False)
lon_bin = pd.cut(df['Longitude'], bins=10, labels=False)
groups = (lat_bin.astype(str) + '_' + lon_bin.astype(str))
# 用一组 holdout groups
uniq = groups.unique()
rng = np.random.RandomState(RANDOM_STATE)
hold = set(rng.choice(uniq, size=max(2, len(uniq) // 5), replace=False))
mask = groups.isin(hold)
Xtr_g, Xte_g = X.loc[~mask], X.loc[mask]
ytr_g, yte_g = y.loc[~mask], y.loc[mask]
model.fit(Xtr_g, ytr_g)
pred_g = model.predict(Xte_g)
print('SPATIAL MAE', mean_absolute_error(yte_g, pred_g),
      'RMSE', mean_squared_error(yte_g, pred_g, squared=False))
print('holdout groups', len(hold), 'test size', mask.sum())


## 4. QuickBaseline（随机划分，便于对比模型）


In [ ]:
from wanglaoshi import CompetitionTools as CT

# 为速度可抽样
if QUICK:
    sample_idx = Xtr_r.sample(min(4000, len(Xtr_r)), random_state=RANDOM_STATE).index
    Xtr_b, ytr_b = Xtr_r.loc[sample_idx], ytr_r.loc[sample_idx]
else:
    Xtr_b, ytr_b = Xtr_r, ytr_r

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_b, ytr_b, Xte_r, yte_r, task_type='regression')
comparison = baseline.compare_models(results, top_n=5)
comparison


## 5. 分房价区间看误差


In [ ]:
best = comparison.iloc[0]['Model']
pred = results[best]['predictions']
err = pd.DataFrame({'y': yte_r.to_numpy(), 'pred': np.asarray(pred)})
err['abs_err'] = (err['y'] - err['pred']).abs()
err['bin'] = pd.qcut(err['y'], 4, duplicates='drop')
display(err.groupby('bin', observed=True)['abs_err'].agg(['mean', 'median', 'count']))


## 6. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/06_california_housing'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred)), 'MedHouseVal': yte_r.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] 随机 vs 空间划分对比
- [x] 分位数区间误差
- [ ] GroupKFold 完整报告均值±方差
- [ ] 去掉 Latitude/Longitude 做 ablation（测泄漏程度）
